# SKU matching production readiness - v2

Fixes over v1, driven by the v1 artifact review:

1. **Gold-label audit (new, section 4).** v1 error analysis showed offers like
   'Chicken Popcorn 400g' with gold `CCF` (CHICKEN FRIES) while the model
   predicted `CKPC` (CHICKEN POP-CORN). Those pairs are audited and metrics are
   reported both on ALL pairs and on the CLEAN subset.
2. **Pipeline-canonical text (fixed, section 6).** v1 evaluated on notebook-built
   text. v2 replicates `src/sku_mapping/embedding/text.py` (version 2.0.0)
   normalization and `label=value` construction, and A/Bs it against v1 text.
3. **RapidFuzz marginal-value analysis (new, section 9).** The business case for
   running the embedding parallel to fuzzy matching: of offers RapidFuzz top-k
   misses, how many does the embedding top-k recover, and what does the UNION buy?
4. **Honest threshold calibration (fixed, section 12).** v1 accepted an operating
   point supported by n=2 dev offers (precision 1.0 -> 0.556 held out). v2
   requires `MIN_ACCEPTS` supporting offers or declares the threshold NOT VIABLE
   and refuses to export an auto-accept score.
5. **Environment reconciled (fixed, section 1).** CPU-friendly, Python 3.11+,
   `sentence-transformers>=3,<6` to match the repo's optional `[embedding]` extra.

The intended role under evaluation is **candidate recall channel** (shortlist
generator parallel to RapidFuzz), NOT an autonomous decider. v1 already showed
the decider role fails open-set separation and threshold generalization.

## 1. Environment (reconciled with the repo)

In [ ]:
# Matches pyproject.toml: python >=3.11, sentence-transformers >=3,<6, CPU ok.
import importlib, subprocess, sys

def ensure(pkg, spec):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', spec])

ensure('sentence_transformers', 'sentence-transformers>=3,<6')
ensure('rapidfuzz', 'rapidfuzz')
ensure('pyarrow', 'pyarrow')
ensure('openpyxl', 'openpyxl')

import hashlib, json, os, platform, re, time, unicodedata
import numpy as np
import pandas as pd

assert sys.version_info >= (3, 11), 'repo targets Python >=3.11'

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

SEED = 42
rng = np.random.default_rng(SEED)
print(f'python {platform.python_version()}   device {DEVICE}')

## 2. Paths

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/promostrater'
except ImportError:
    PROJECT_DIR = os.environ.get('PROMOSTRATER_DIR', '.')

MASTER_PATH = os.path.join(PROJECT_DIR, 'Product_Master.xlsx')
PAIRS_PATH = os.path.join(PROJECT_DIR, 'data', 'processed', 'training_features.parquet')
OFFERS_DUMP_PATH = os.path.join(PROJECT_DIR, 'Alkabeer_Export_Data_Clickflyer.csv')
OUT_DIR = os.path.join(PROJECT_DIR, 'prod_eval_v2')
CACHE_DIR = os.path.join(PROJECT_DIR, 'emb_cache_v2')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

for lbl, p in (('master', MASTER_PATH), ('pairs ', PAIRS_PATH), ('offers', OFFERS_DUMP_PATH)):
    ok = os.path.exists(p)
    size = os.path.getsize(p) / 1e6 if ok else 0
    print(f'{lbl}  {"OK " if ok else "MISSING"}  {p}  {size:.1f} MB')

# Guard against the truncated-copy failure seen with a 0-byte parquet:
assert os.path.getsize(PAIRS_PATH) > 1000, 'training_features.parquet is empty/corrupt - re-copy it'
with open(PAIRS_PATH, 'rb') as fh:
    assert fh.read(4) == b'PAR1', 'not a valid parquet file (missing PAR1 magic)'

## 3. Load data

In [ ]:
master_raw = pd.read_excel(MASTER_PATH)
pairs = pd.read_parquet(PAIRS_PATH)

master = pd.DataFrame({
    'master_itemcode': master_raw['Itemcode'].astype(str),
    'master_item_description': master_raw['Itemname'].astype(str),
    'master_item_family': master_raw['Item-Cat-4'].astype(str),
    'master_item_category': master_raw['Item-Cat-2'].astype(str),
    'master_item_long_description': master_raw['Item Description'].astype(str),
    'master_item_spec': master_raw['Item-Spec'].astype(str),
}).drop_duplicates('master_itemcode').reset_index(drop=True)

gold = (pairs[pairs['pair_label'] == 1][['offer_text', 'master_itemcode']]
        .dropna().drop_duplicates().reset_index(drop=True))
n_before = len(gold)
gold['master_itemcode'] = gold['master_itemcode'].astype(str)
gold = gold[gold['master_itemcode'].isin(set(master['master_itemcode']))]
gold = gold.reset_index(drop=True)

n_master = len(master)
print(f'master SKUs:        {n_master:,}')
print(f'gold pairs:         {len(gold):,}  (dropped {n_before - len(gold):,})')
print(f'distinct offers:    {gold["offer_text"].nunique():,}')
print(f'distinct gold SKUs: {gold["master_itemcode"].nunique():,}')
if n_master < 1000:
    print('\nNOTE: small catalogue - recall numbers are optimistic; see section 13.')

## 4. Gold-label audit (NEW - fixes v1 issue #1)

Two independent detectors, no embedding involved (so the audit cannot be
circular):

- **Lexical contradiction**: RapidFuzz similarity of the offer against its GOLD
  master name is much lower than against its best NON-gold master name. The v1
  popcorn->fries pairs are exactly this shape.
- **Inconsistent labelling**: near-identical normalized offer texts mapped to
  different gold SKUs.

Flagged pairs are exported for a human verdict, and every retrieval metric
below is reported on ALL pairs and on the CLEAN subset. Suspects are NOT
deleted - being flagged is evidence, not proof: some mappings encode real
business knowledge the text does not carry (e.g. the retailer's 'popcorn'
listing may genuinely be the fries SKU).

In [ ]:
from rapidfuzz import fuzz

def lex_norm(s):
    s = unicodedata.normalize('NFKC', str(s)).lower()
    s = re.sub(r'[^a-z0-9 ]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

master_names = master['master_item_description'].map(lex_norm).tolist()
code_index = {c: i for i, c in enumerate(master['master_itemcode'])}
master_codes = master['master_itemcode'].to_numpy()

offer_norm = gold['offer_text'].map(lex_norm).tolist()
gold_positions = np.array([code_index[c] for c in gold['master_itemcode']])

gold_lex = np.zeros(len(gold))
best_other_lex = np.zeros(len(gold))
best_other_idx = np.zeros(len(gold), dtype=int)
for i, otext in enumerate(offer_norm):
    scores = np.array([fuzz.token_set_ratio(otext, m) for m in master_names])
    gold_lex[i] = scores[gold_positions[i]]
    scores[gold_positions[i]] = -1
    best_other_idx[i] = int(scores.argmax())
    best_other_lex[i] = scores.max()

LEX_GAP = 25.0   # gold must not trail the best non-gold name by more than this
suspect_lex = (best_other_lex - gold_lex) >= LEX_GAP

# Inconsistent labelling: same normalized offer text, different gold SKUs.
text_to_golds = gold.assign(norm=offer_norm).groupby('norm')['master_itemcode'].nunique()
inconsistent_texts = set(text_to_golds[text_to_golds > 1].index)
suspect_inc = np.array([t in inconsistent_texts for t in offer_norm])

SUSPECT = suspect_lex | suspect_inc
CLEAN = ~SUSPECT

audit = pd.DataFrame({
    'offer_text': gold['offer_text'],
    'gold_itemcode': gold['master_itemcode'],
    'gold_name': [master_names[i] for i in gold_positions],
    'gold_lex_score': gold_lex,
    'best_other_itemcode': master_codes[best_other_idx],
    'best_other_name': [master_names[i] for i in best_other_idx],
    'best_other_lex_score': best_other_lex,
    'flag_lexical_contradiction': suspect_lex,
    'flag_inconsistent_labelling': suspect_inc,
    'human_verdict': '',   # fill: gold_correct | gold_wrong | ambiguous
})
audit[SUSPECT].to_csv(f'{OUT_DIR}/suspect_gold_pairs.csv', index=False)

print(f'suspect pairs: {SUSPECT.sum():,} of {len(gold):,} ({SUSPECT.mean():.1%})')
print(f'  lexical contradiction: {suspect_lex.sum():,}')
print(f'  inconsistent labelling: {suspect_inc.sum():,}')
print(f'clean pairs: {CLEAN.sum():,}')
print(f'\n-> {OUT_DIR}/suspect_gold_pairs.csv  (fill human_verdict)')
print('\nworst lexical contradictions:')
print(audit[suspect_lex].sort_values('gold_lex_score')
      .head(10)[['offer_text', 'gold_name', 'best_other_name']].to_string(index=False))

## 5. Model

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-small-en-v1.5'
MODEL_CARD = {
    'name': MODEL_NAME,
    'embedding_dim': 384,
    'max_seq_length': 512,
    'recommended_similarity': 'cosine',
    'license': 'MIT',
    'language': 'English only',
}
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(json.dumps(MODEL_CARD, indent=2))

## 6. Pipeline-canonical text construction (FIXED - v1 issue #2)

Replicates `src/sku_mapping/embedding/text.py` TEXT_CONSTRUCTION_VERSION 2.0.0:
same normalization, same `label=value | ...` layout. If the repo version
changes, re-sync this cell and clear the cache. The v1 notebook text is kept
for a like-for-like A/B in section 8.

In [ ]:
TEXT_CONSTRUCTION_VERSION = '2.0.0'  # keep in sync with the repo

def normalize_embedding_text(value):
    text = unicodedata.normalize('NFKC', '' if value is None else str(value)).lower()
    text = text.replace('\u00d7', ' x ')
    text = re.sub(r'\bgrams?\b|\bgms?\b|\bgm\b', 'g', text)
    text = re.sub(r'\bkilograms?\b|\bkgs?\b', 'kg', text)
    text = re.sub(r'(\d)\s*kg\b', r'\1 kg', text)
    text = re.sub(r'(\d)\s*g\b', r'\1 g', text)
    text = re.sub(r'(?<=\d),(?=\d)', '', text)
    text = re.sub(r'[,()]+', ' ', text)
    text = re.sub(r'[;:]+', ' ', text)
    text = re.sub(r'\s*[|]\s*', ' | ', text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip(' |.,')

def parts(fields, row):
    out = []
    for label, column in fields:
        value = normalize_embedding_text(row.get(column, ''))
        if value and value != 'nan':
            out.append(f'{label}={value}')
    return ' | '.join(out)

MASTER_FIELDS = (
    ('brand', 'master_brand'),
    ('item', 'master_item_description'),
    ('family', 'master_item_family'),
    ('category', 'master_item_category'),
    ('description', 'master_item_long_description'),
    ('pack', 'master_item_spec'),
)
OFFER_FIELDS = (
    ('brand', 'offer_brand'),
    ('offer', 'offer_text'),
)

master['master_brand'] = 'Al Kabeer'
master_texts = [parts(MASTER_FIELDS, row) for row in master.to_dict('records')]

gold_rows = gold.assign(offer_brand='Al Kabeer').to_dict('records')
offer_texts = [parts(OFFER_FIELDS, row) for row in gold_rows]

# v1-style text for the A/B (raw offer_text as v1 used it)
offer_texts_v1 = gold['offer_text'].tolist()

print('pipeline master text example:\n ', master_texts[0][:160])
print('pipeline offer  text example:\n ', offer_texts[0][:160])

from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
_lens = np.array([len(_tok.encode(t)) for t in master_texts])
over = float((_lens > MODEL_CARD['max_seq_length']).mean())
print(f'\nmaster texts over max_seq_length: {over:.1%}')

## 7. Encoding with versioned cache

In [ ]:
def _key(texts, tag):
    h = hashlib.sha256()
    h.update(MODEL_NAME.encode())
    h.update(TEXT_CONSTRUCTION_VERSION.encode())
    h.update(str(MODEL_CARD['embedding_dim']).encode())
    h.update(tag.encode())
    for t in texts:
        h.update(t.encode('utf-8', 'replace'))
    return h.hexdigest()[:24]

def encode(texts, mdl, tag, batch_size=64, use_cache=True):
    path = os.path.join(CACHE_DIR, f'{tag}_{_key(texts, tag)}.npy')
    if use_cache and os.path.exists(path):
        return np.load(path)
    vec = mdl.encode(list(texts), batch_size=batch_size, convert_to_numpy=True,
                     show_progress_bar=True)
    if use_cache:
        np.save(path, vec)
    return vec

def l2norm(x):
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)

mas_vec = l2norm(encode(master_texts, model, 'master_pipeline'))
off_vec = l2norm(encode(offer_texts, model, 'offer_pipeline'))
off_vec_v1 = l2norm(encode(offer_texts_v1, model, 'offer_v1style'))
print('master', mas_vec.shape, '  offers', off_vec.shape)

## 8. Retrieval metrics: ALL vs CLEAN, pipeline vs v1 text

k=10 is now first-class: the intended role is a recall channel and the
downstream stack (19-column features + LightGBM + agreement policy) does the
deciding.

In [ ]:
def ranks_from_sims(sim, gp):
    order = np.argsort(-sim, axis=1)
    return (order == gp[:, None]).argmax(axis=1) + 1

def metrics(ranks, ks=(1, 3, 5, 10)):
    return {f'recall@{k}': float((ranks <= k).mean()) for k in ks}

def ci(ranks, k=10, n_boot=5000, seed=0):
    r = np.random.default_rng(seed)
    hits = (ranks <= k).astype(float)
    boots = [hits[r.integers(0, len(hits), len(hits))].mean() for _ in range(n_boot)]
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

SIM = off_vec @ mas_vec.T
SIM_V1 = off_vec_v1 @ mas_vec.T
RANKS = ranks_from_sims(SIM, gold_positions)
RANKS_V1 = ranks_from_sims(SIM_V1, gold_positions)

rows = []
for name, ranks, mask in (
    ('pipeline text / ALL', RANKS, np.ones(len(gold), bool)),
    ('pipeline text / CLEAN', RANKS, CLEAN),
    ('v1 text / ALL', RANKS_V1, np.ones(len(gold), bool)),
    ('v1 text / CLEAN', RANKS_V1, CLEAN),
):
    m = metrics(ranks[mask])
    lo, hi = ci(ranks[mask])
    rows.append({'setup': name, 'n': int(mask.sum()), **m,
                 'recall@10_ci': f'[{lo:.3f}, {hi:.3f}]'})
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))
print('\nIf CLEAN >> ALL, a chunk of the v1 error rate was label noise, not model error.')
print('If pipeline text >> v1 text, the v1 numbers understated the deployed model.')

## 9. RapidFuzz marginal-value analysis (NEW - v1 issue #3)

The single number that justifies (or kills) 'embedding parallel to fuzzy':
**of the offers RapidFuzz top-k misses, what fraction does embedding top-k
recover?** Uses `token_set_ratio` like the production candidate generator
(category gating and pack bonuses omitted - both would only help fuzzy, so the
union uplift here is a fair, conservative estimate of what the embedding adds).

In [ ]:
K_LIST = (5, 10)

fuzzy_scores = np.zeros((len(gold), n_master))
for i, otext in enumerate(offer_norm):
    fuzzy_scores[i] = [fuzz.token_set_ratio(otext, m) for m in master_names]
FUZZY_RANKS = ranks_from_sims(fuzzy_scores, gold_positions)

def topk_sets(score_matrix, k):
    return np.argsort(-score_matrix, axis=1)[:, :k]

print(f'{"k":>3} {"fuzzy":>7} {"embed":>7} {"union":>7} '
      f'{"fuzzy misses":>13} {"recovered":>10} {"recovery":>9}')
marginal = []
for k in K_LIST:
    fz_hit = FUZZY_RANKS <= k
    em_hit = RANKS <= k
    fz_top = topk_sets(fuzzy_scores, k)
    em_top = topk_sets(SIM, k)
    union_hit = np.array([
        gold_positions[i] in set(fz_top[i]) | set(em_top[i])
        for i in range(len(gold))
    ])
    misses = ~fz_hit
    recovered = int((misses & em_hit).sum())
    recovery = recovered / max(int(misses.sum()), 1)
    marginal.append({'k': k,
                     'fuzzy_recall': float(fz_hit.mean()),
                     'embed_recall': float(em_hit.mean()),
                     'union_recall': float(union_hit.mean()),
                     'fuzzy_misses': int(misses.sum()),
                     'recovered_by_embedding': recovered,
                     'recovery_rate': float(recovery)})
    print(f'{k:>3} {fz_hit.mean():>7.4f} {em_hit.mean():>7.4f} '
          f'{union_hit.mean():>7.4f} {int(misses.sum()):>13,} '
          f'{recovered:>10,} {recovery:>9.1%}')
marginal_df = pd.DataFrame(marginal)
marginal_df.to_csv(f'{OUT_DIR}/fuzzy_vs_embedding_marginal_value.csv', index=False)

k = 10
uplift = float(marginal_df.loc[marginal_df.k == k, 'union_recall'].iloc[0]
               - marginal_df.loc[marginal_df.k == k, 'fuzzy_recall'].iloc[0])
print(f'\nunion uplift over fuzzy alone at k={k}: {uplift:+.4f}')
print('If this uplift is near zero, re-positioning the embedding buys nothing')
print('and the integration cost is not justified. That is the decision number.')

## 10. Grouped held-out split

In [ ]:
def grouped_split(groups, frac=0.5, seed=SEED):
    r = np.random.default_rng(seed)
    uniq = np.unique(groups)
    r.shuffle(uniq)
    dev_groups = set(uniq[: int(len(uniq) * frac)].tolist())
    dev = np.array([g in dev_groups for g in groups])
    return dev, ~dev

DEV, TEST = grouped_split(gold_positions)
assert not (set(gold_positions[DEV]) & set(gold_positions[TEST]))
print(f'dev  offers {DEV.sum():,}   SKUs {len(set(gold_positions[DEV]))}')
print(f'test offers {TEST.sum():,}   SKUs {len(set(gold_positions[TEST]))}')
print(f'\ndev  recall@10 {(RANKS[DEV] <= 10).mean():.4f}')
print(f'test recall@10 {(RANKS[TEST] <= 10).mean():.4f}')

## 11. Open-set behaviour

In [ ]:
top1_idx = SIM.argmax(axis=1)
top1_score = SIM[np.arange(len(SIM)), top1_idx]
top1_correct = top1_idx == gold_positions

_masked = SIM.copy()
_masked[np.arange(len(SIM)), gold_positions] = -np.inf
nomatch_score = _masked.max(axis=1)

sep = float(np.percentile(top1_score[top1_correct], 10)
            - np.percentile(nomatch_score, 90))
print(f'correct top-1 mean {top1_score[top1_correct].mean():.4f}')
print(f'no-match  mean     {nomatch_score.mean():.4f}')
print(f'separation (correct p10 - nomatch p90): {sep:+.4f}')
if sep <= 0:
    print('\nNEGATIVE separation: cosine score CANNOT reject unmatched offers.')
    print('This alone disqualifies the autonomous-decider role and is why v2')
    print('evaluates the recall-channel role instead.')

## 12. Honest threshold calibration (FIXED - v1 issue #4)

v1 chose a threshold supported by 2 dev offers (precision 1.0 on dev, 0.556
held out). v2 requires at least `MIN_ACCEPTS` accepted dev offers before an
operating point counts as viable, and exports NO auto-accept threshold
otherwise.

In [ ]:
TARGET_PRECISION = 0.98
MIN_ACCEPTS = 30

def operating_point(t, closed_score, correct, open_score):
    accepted = closed_score >= t
    false_open = int((open_score >= t).sum())
    n_acc = int(accepted.sum()) + false_open
    tp = int((accepted & correct).sum())
    total = len(closed_score) + len(open_score)
    return {'threshold': float(t),
            'precision': float(tp / n_acc) if n_acc else np.nan,
            'automation_rate': float(n_acc / total),
            'n_accepted': n_acc}

grid = np.unique(np.concatenate([top1_score[DEV], nomatch_score[DEV]]))
dev_df = pd.DataFrame([operating_point(t, top1_score[DEV], top1_correct[DEV],
                                       nomatch_score[DEV]) for t in grid])
viable = dev_df[(dev_df.precision >= TARGET_PRECISION)
                & (dev_df.n_accepted >= MIN_ACCEPTS)]

if viable.empty:
    ACCEPT_T = None
    print(f'NO viable operating point: nothing reaches {TARGET_PRECISION:.0%} '
          f'precision with >= {MIN_ACCEPTS} dev accepts.')
    print('AUTO-ACCEPT IS NOT VIABLE for this model on this data. No threshold')
    print('will be exported. Use the model as a recall channel only.')
else:
    chosen = viable.sort_values('threshold').iloc[0]
    ACCEPT_T = float(chosen['threshold'])
    test_op = operating_point(ACCEPT_T, top1_score[TEST], top1_correct[TEST],
                              nomatch_score[TEST])
    print(f'ACCEPT threshold {ACCEPT_T:.4f}  dev precision {chosen.precision:.3f} '
          f'(n={int(chosen.n_accepted)})  test precision {test_op["precision"]:.3f} '
          f'(n={test_op["n_accepted"]})')

dev_df.to_csv(f'{OUT_DIR}/threshold_curve_dev.csv', index=False)

## 13. Catalogue scaling with extrapolation

In [ ]:
def scaling_point(sim, gp, pool_n, trials=8, seed=0, k=10, min_offers=10):
    r = np.random.default_rng(seed)
    vals = []
    for _ in range(trials):
        keep = r.choice(sim.shape[1], size=pool_n, replace=False)
        keep_set = set(keep.tolist())
        mask = np.array([g in keep_set for g in gp])
        if mask.sum() < min_offers:
            continue
        pos = {m: i for i, m in enumerate(keep)}
        sub = sim[mask][:, keep]
        sub_gold = np.array([pos[int(g)] for g in gp[mask]])
        vals.append((ranks_from_sims(sub, sub_gold) <= k).mean())
    return float(np.mean(vals)) if vals else np.nan

sizes = [s for s in (25, 50, 100, 200) if s < n_master] + [n_master]
curve = pd.DataFrame([{'catalogue_size': s,
                       'recall@10': scaling_point(SIM, gold_positions, s, seed=11),
                       'recall@5': scaling_point(SIM, gold_positions, s, seed=11, k=5)}
                      for s in sizes])
print(curve.to_string(index=False))

fit = curve.dropna()
slope = float(np.polyfit(np.log(fit['catalogue_size']), fit['recall@10'], 1)[0])
r10 = float(curve['recall@10'].iloc[-1])
for target in (500, 1000):
    est = r10 + slope * np.log(target / n_master)
    print(f'extrapolated recall@10 at {target} SKUs: ~{est:.3f}')
print('\nExtrapolation is a straight line in log-size: a planning number, not a promise.')
curve.to_csv(f'{OUT_DIR}/catalogue_scaling_curve.csv', index=False)

## 14. Error analysis with label noise separated

In [ ]:
err = pd.DataFrame({
    'offer_text': gold['offer_text'],
    'gold_itemcode': gold['master_itemcode'],
    'predicted': master_codes[top1_idx],
    'predicted_text': [master_texts[i][:70] for i in top1_idx],
    'gold_text': [master_texts[i][:70] for i in gold_positions],
    'top1_score': top1_score,
    'gold_rank': RANKS,
    'correct': top1_correct,
    'suspect_gold': SUSPECT,
})
err.to_csv(f'{OUT_DIR}/error_analysis.csv', index=False)

clean_err = err[CLEAN & ~err['correct']]
print(f'top-1 errors ALL:   {int((~err.correct).sum()):,} of {len(err):,} '
      f'({(~err.correct).mean():.1%})')
print(f'top-1 errors CLEAN: {len(clean_err):,} of {int(CLEAN.sum()):,} '
      f'({len(clean_err)/max(int(CLEAN.sum()),1):.1%})')
print('\nworst CLEAN failures (gold ranked lowest):')
print(clean_err.sort_values('gold_rank', ascending=False)
      .head(10)[['offer_text', 'predicted_text', 'gold_text', 'gold_rank']]
      .to_string(index=False))

## 15. Go / no-go checklist v2

In [ ]:
checks = []
def chk(name, passed, detail):
    status = 'PASS' if passed is True else ('ATTENTION' if passed is None else 'FAIL')
    checks.append({'check': name, 'status': status, 'detail': detail})

m_all = metrics(RANKS)
uplift10 = float(marginal_df.loc[marginal_df.k == 10, 'union_recall'].iloc[0]
                 - marginal_df.loc[marginal_df.k == 10, 'fuzzy_recall'].iloc[0])
rec10 = float(marginal_df.loc[marginal_df.k == 10, 'recovery_rate'].iloc[0])

chk('Tested on pipeline text construction', True,
    f'TEXT_CONSTRUCTION_VERSION {TEXT_CONSTRUCTION_VERSION}')
chk('Gold labels audited', None if SUSPECT.sum() else True,
    f'{int(SUSPECT.sum())} suspects exported - human verdicts pending')
chk('Recall-channel value demonstrated', True if uplift10 >= 0.02 else None,
    f'union uplift over fuzzy at k=10: {uplift10:+.4f}, recovery {rec10:.1%}')
chk('Shortlist quality (recall@10, all pairs)', m_all['recall@10'] >= 0.85,
    f'{m_all["recall@10"]:.3f}')
chk('Auto-accept viable', False if ACCEPT_T is None else True,
    'no threshold meets precision with adequate support' if ACCEPT_T is None
    else f'threshold {ACCEPT_T:.4f}')
chk('Unmatched offers separable', sep > 0, f'separation {sep:+.4f}')
chk('No silent truncation', over <= 0.01, f'{over:.1%} of master texts truncated')
chk('Catalogue is full size', None if n_master < 1000 else True,
    f'{n_master} SKUs - confirm this is the whole catalogue')
chk('Environment matches repo constraints', True,
    'python>=3.11, sentence-transformers>=3,<6, CPU-capable')

status = pd.DataFrame(checks)
print(status.to_string(index=False))
status.to_csv(f'{OUT_DIR}/go_no_go_checklist.csv', index=False)
print('\nExpected verdict: NO-GO as decider (by design). GO/NO-GO as recall')
print('channel is decided by the section 9 uplift number plus the label audit.')

## 16. Export production spec v2

In [ ]:
spec = {
    'role': 'candidate_recall_channel',
    'role_note': ('Shortlist generator parallel to RapidFuzz. NOT an autonomous '
                  'decider: open-set separation and threshold generalization '
                  'both failed in v1 and are re-tested here.'),
    'model': MODEL_CARD,
    'text_construction_version': TEXT_CONSTRUCTION_VERSION,
    'config': {
        'backend': 'local_sentence_transformer',
        'device_evaluated': DEVICE,
        'normalize_vectors': 1,
        'similarity_metric': 'cosine',
        'shortlist_k': 10,
    },
    'thresholds': ({'auto_accept_min_score': ACCEPT_T,
                    'min_supporting_accepts': MIN_ACCEPTS}
                   if ACCEPT_T is not None else
                   {'auto_accept': 'NOT_VIABLE',
                    'reason': f'no threshold reaches {TARGET_PRECISION:.0%} '
                              f'precision with >={MIN_ACCEPTS} dev accepts'}),
    'evaluation': {
        'n_gold_pairs': int(len(gold)),
        'n_suspect_gold_pairs': int(SUSPECT.sum()),
        'catalogue_size': int(n_master),
        'retrieval_all': metrics(RANKS),
        'retrieval_clean': metrics(RANKS[CLEAN]),
        'open_set_separation': float(sep),
        'fuzzy_vs_embedding_marginal': marginal_df.to_dict('records'),
    },
    'environment': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'device': DEVICE,
    },
    'known_limitations': [
        'Gold-label audit verdicts pending human review of suspect_gold_pairs.csv.',
        'Fuzzy baseline omits category gating and pack bonuses from the production candidate generator.',
        f'Catalogue was {n_master} SKUs; see scaling extrapolation before trusting recall at production size.',
        'Open-set simulated by gold removal; genuinely foreign products may score differently.',
        'Model is English-only. Non-English supplier text is out of distribution.',
        'Clear emb_cache_v2 on any model, dimension, or text-construction change.',
    ],
}

with open(f'{OUT_DIR}/production_spec_v2.json', 'w') as f:
    json.dump(spec, f, indent=2)
print(json.dumps(spec['thresholds'], indent=2))
print(f'\nArtifacts in {OUT_DIR}:')
for name in sorted(os.listdir(OUT_DIR)):
    print(' ', name)